# Exam Difficulty Analysis

## Project Overview
Analyze question-level and exam-level data to study score distributions, identify hard topics, and evaluate assessment quality signals such as discrimination indices and reliability.

## Learning Objectives
- Analyze score distributions at question and exam level
- Identify difficult questions and topic areas
- Calculate item difficulty and discrimination indices
- Assess overall exam quality using classical test theory metrics

## Problem Statement
Assessment designers need to understand which questions are too hard, too easy, or poorly discriminating so they can improve exam quality, ensure fair grading, and align assessments with learning objectives.

## Why This Project Matters
Well-designed assessments are the backbone of fair evaluation. Poor questions introduce noise, penalize well-prepared students, and undermine trust in the evaluation system. Data-driven item analysis catches these problems.

## Dataset Overview
Synthetic exam dataset: 500 students × 40 questions across 5 topics, with per-question scores (0/1) and total exam scores.

## Dataset Source and License Notes
- Synthetic data generated for educational analysis
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
N_STUDENTS = 500
N_QUESTIONS = 40
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
topics = ['Algebra', 'Geometry', 'Statistics', 'Calculus', 'Probability']
q_per_topic = N_QUESTIONS // len(topics)  # 8 each

# Question difficulty (probability of getting it right)
q_difficulty = np.concatenate([
    np.random.uniform(0.3, 0.9, q_per_topic) for _ in topics
])
q_topic = np.repeat(topics, q_per_topic)
q_names = [f'Q{i+1}' for i in range(N_QUESTIONS)]

# Student ability ~ N(0.6, 0.15)
ability = np.clip(np.random.normal(0.6, 0.15, N_STUDENTS), 0.1, 0.95)

# Generate responses: P(correct) = ability * difficulty_weight
responses = np.zeros((N_STUDENTS, N_QUESTIONS), dtype=int)
for i in range(N_STUDENTS):
    for j in range(N_QUESTIONS):
        p = ability[i] * q_difficulty[j]
        responses[i, j] = np.random.binomial(1, np.clip(p, 0.05, 0.95))

df_resp = pd.DataFrame(responses, columns=q_names)
df_resp.insert(0, 'StudentID', [f'S{i:04d}' for i in range(N_STUDENTS)])
df_resp['TotalScore'] = df_resp[q_names].sum(axis=1)
df_resp['Percentage'] = (df_resp['TotalScore'] / N_QUESTIONS * 100).round(1)

# Question metadata
df_questions = pd.DataFrame({
    'Question': q_names,
    'Topic': q_topic,
    'DesignedDifficulty': q_difficulty.round(3)
})

print(f'Response matrix: {df_resp.shape}')
print(f'Score range: {df_resp["TotalScore"].min()} — {df_resp["TotalScore"].max()} out of {N_QUESTIONS}')
print(f'Mean score: {df_resp["TotalScore"].mean():.1f} ({df_resp["Percentage"].mean():.1f}%)')
df_resp.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df_resp.isnull().sum().sum()}')
print(f'Students: {len(df_resp)}')
print(f'Questions: {N_QUESTIONS}')
print(f'Topics: {df_questions["Topic"].nunique()}')

## Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_resp['Percentage'].hist(bins=20, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].axvline(df_resp['Percentage'].mean(), color='red', linestyle='--', label=f'Mean: {df_resp["Percentage"].mean():.1f}%')
axes[0].set_title('Exam Score Distribution')
axes[0].set_xlabel('Score (%)')
axes[0].legend()

df_resp['Percentage'].plot.box(ax=axes[1])
axes[1].set_title('Score Box Plot')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'score_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Skewness: {df_resp["Percentage"].skew():.3f}')
print(f'Kurtosis: {df_resp["Percentage"].kurtosis():.3f}')

## Item Difficulty Index (p-value)

In [ ]:
# p-value = proportion of students who got it right
p_values = df_resp[q_names].mean()
df_questions['ObservedDifficulty'] = p_values.values

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['red' if p < 0.3 else 'orange' if p < 0.5 else 'green' if p <= 0.8 else 'blue' for p in p_values]
ax.bar(q_names, p_values, color=colors, edgecolor='black')
ax.axhline(0.3, color='red', linestyle='--', alpha=0.5, label='Hard threshold (0.3)')
ax.axhline(0.8, color='blue', linestyle='--', alpha=0.5, label='Easy threshold (0.8)')
ax.set_title('Item Difficulty Index (p-value) per Question')
ax.set_ylabel('Proportion Correct')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'item_difficulty.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Hard questions (p < 0.3): {(p_values < 0.3).sum()}')
print(f'Easy questions (p > 0.8): {(p_values > 0.8).sum()}')
print(f'Ideal range (0.3-0.8): {((p_values >= 0.3) & (p_values <= 0.8)).sum()}')

## Item Discrimination Index

In [ ]:
# Point-biserial correlation: correlation of each item with total score (excluding that item)
disc_indices = []
for q in q_names:
    other_score = df_resp['TotalScore'] - df_resp[q]
    disc_indices.append(df_resp[q].corr(other_score))
df_questions['Discrimination'] = disc_indices

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['red' if d < 0.2 else 'orange' if d < 0.3 else 'green' for d in disc_indices]
ax.bar(q_names, disc_indices, color=colors, edgecolor='black')
ax.axhline(0.2, color='red', linestyle='--', alpha=0.5, label='Poor discrimination (0.2)')
ax.axhline(0.3, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.3)')
ax.set_title('Item Discrimination Index per Question')
ax.set_ylabel('Point-Biserial Correlation')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'item_discrimination.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Poor discriminators (< 0.2): {(df_questions["Discrimination"] < 0.2).sum()}')
print(f'Good discriminators (>= 0.3): {(df_questions["Discrimination"] >= 0.3).sum()}')

## Topic-Level Difficulty

In [ ]:
topic_stats = df_questions.groupby('Topic').agg(
    MeanDifficulty=('ObservedDifficulty', 'mean'),
    MeanDiscrimination=('Discrimination', 'mean'),
    NumQuestions=('Question', 'count')
).round(3)
print('Topic-Level Summary:')
print(topic_stats)

fig, ax = plt.subplots(figsize=(10, 5))
topic_stats['MeanDifficulty'].sort_values().plot.barh(ax=ax, edgecolor='black', color='teal')
ax.set_title('Average Difficulty by Topic')
ax.set_xlabel('Mean p-value (higher = easier)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'topic_difficulty.png'), dpi=100, bbox_inches='tight')
plt.show()

## Difficulty vs Discrimination Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for topic in topics:
    mask = df_questions['Topic'] == topic
    ax.scatter(df_questions.loc[mask, 'ObservedDifficulty'],
               df_questions.loc[mask, 'Discrimination'],
               label=topic, s=60, alpha=0.8)
ax.set_xlabel('Item Difficulty (p-value)')
ax.set_ylabel('Discrimination Index')
ax.set_title('Item Difficulty vs Discrimination')
ax.axhline(0.2, color='red', linestyle='--', alpha=0.4)
ax.axvline(0.3, color='gray', linestyle='--', alpha=0.4)
ax.axvline(0.8, color='gray', linestyle='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'diff_vs_disc.png'), dpi=100, bbox_inches='tight')
plt.show()

## Exam Reliability (KR-20)

In [ ]:
# Kuder-Richardson 20 for binary items
k = N_QUESTIONS
p_vals = df_resp[q_names].mean().values
q_vals = 1 - p_vals
total_var = df_resp['TotalScore'].var()
kr20 = (k / (k - 1)) * (1 - np.sum(p_vals * q_vals) / total_var)
print(f'KR-20 Reliability: {kr20:.3f}')
if kr20 >= 0.80:
    print('Interpretation: Good to excellent reliability')
elif kr20 >= 0.70:
    print('Interpretation: Acceptable reliability')
else:
    print('Interpretation: Below acceptable — exam may need revision')

## Interpretation and Business Insight
- **Score distribution** shape reveals whether the exam is well-calibrated for the student population
- **Item difficulty** flags questions that are too easy (everyone gets them) or too hard (nearly nobody does) — both add noise, not signal
- **Discrimination index** identifies questions that don't differentiate strong from weak students — these should be revised or replaced
- **Topic-level analysis** shows which subject areas are hardest, guiding curriculum reinforcement
- **KR-20 reliability** measures internal consistency — exams below 0.7 should be reviewed
- The best items sit in the 0.3-0.8 difficulty range with discrimination > 0.3

## Limitations
- Synthetic data with simplified item response model
- No partial credit or multi-point questions
- No distractor analysis for multiple-choice items
- No item response theory (IRT) modeling
- No cross-exam comparability

## How to Improve This Project
- Add IRT (Item Response Theory) modeling for deeper psychometric analysis
- Include distractor analysis for multiple-choice items
- Track item performance across multiple exam sittings
- Build automated item flagging pipelines
- Add differential item functioning (DIF) analysis for fairness

## Production Considerations
- Automated post-exam item analysis reports
- Item bank management with difficulty/discrimination metadata
- Adaptive testing systems using calibrated item parameters
- Real-time exam analytics for proctored online assessments

## Common Mistakes
- Using only difficulty without discrimination to evaluate items
- Ignoring that very easy items may still be important for confidence building
- Treating all low-discrimination items as bad — some may test unique important skills
- Not recalibrating item parameters across different student cohorts

## Mini Challenge / Exercises
1. Which topic has the lowest average discrimination? Should those items be revised?
2. Find all items with p < 0.3 AND discrimination < 0.2 — these are the worst items. How many are there?
3. If you removed all poor discriminators (< 0.2), what would the new KR-20 be?
4. Plot the score distribution separately for the top and bottom quartile students — how much overlap is there?

## Final Summary / Key Takeaways
- Classical item analysis (difficulty + discrimination) is essential for exam quality
- Items in the 0.3-0.8 difficulty range with discrimination > 0.3 are ideal
- KR-20 provides a single reliability number for the entire exam
- Topic-level analysis connects assessment quality to curriculum gaps
- Systematic item analysis improves fairness, reliability, and learning alignment